# Eye-tracking CSV imputation

This notebook runs the released three-stage pipeline on one GazeBase eye-tracking CSV from the locally configured data directory. It uses the recording's existing missing and invalid samples; it never creates an artificial gap or edits the source CSV. A working copy is created in memory, and outputs are written only below this notebook's `outputs/` directory.

The example uses **z-score outlier detection** and **robust standardization**.

In [1]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'pyproject.toml').is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError('Start Jupyter from inside the repository.')
    PROJECT_ROOT = PROJECT_ROOT.parent

# Load simple KEY=VALUE entries without adding an undeclared dependency.
for line in (PROJECT_ROOT / '.env').read_text(encoding='utf-8').splitlines():
    line = line.strip()
    if line and not line.startswith('#') and '=' in line:
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip(chr(34)).strip(chr(39)))

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from gap_imputation_benchmark.algorithm import DomainImputationConfig, RunMetadata, impute_with_rf_selector
from gap_imputation_benchmark.loaders.gazebase import load_gazebase_reading

DATA_ROOT = Path(os.environ['BENCHMARK_DATA_DIR'])
OUTPUT_DIR = PROJECT_ROOT / 'notebooks' / 'algorithm' / 'eyetracking' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXECUTOR_NAME = 'Oliver'
EXECUTOR_INFO_PATH = PROJECT_ROOT / 'data' / 'person_info_examples' / 'executor_example.json'
RESPONSIBLE_PERSON_NAME = 'Jenny'
RESPONSIBLE_PERSON_INFO_PATH = PROJECT_ROOT / 'data' / 'person_info_examples' / 'executor_responsible_person_example.json'
EXECUTION_NOTEBOOK_PATH = PROJECT_ROOT / 'notebooks' / 'algorithm' / 'eyetracking' / '01_eyetracking_csv_imputation.ipynb'
print(f'Repository: {PROJECT_ROOT.name}; data root configured: {DATA_ROOT.is_dir()}')

Repository: github; data root configured: True


In [2]:
# Select one real GazeBase Reading CSV deterministically.
source_files = sorted((DATA_ROOT / 'raw' / 'GazeBase_v2_0').rglob('S_*_TEX.csv'))
if not source_files:
    raise FileNotFoundError('No GazeBase Reading CSV found under BENCHMARK_DATA_DIR/raw/GazeBase_v2_0/.')

SOURCE_CSV = source_files[0]
frame = load_gazebase_reading(SOURCE_CSV)[['timestamp_ms', 'gaze_x', 'is_valid']].copy()
frame = frame.rename(columns={'timestamp_ms': 'timestamp'})
VALUE_COLUMN = 'gaze_x'
frame.head(), SOURCE_CSV.name

(   timestamp     gaze_x  is_valid
 0        0.0 -15.275756      True
 1        1.0 -15.283961      True
 2        2.0 -15.289431      True
 3        3.0 -15.297635      True
 4        4.0 -15.300369      True,
 'S_1001_S1_TEX.csv')

In [3]:
# Use the recording's genuine missing samples; invalid samples are not observable context.
frame.loc[~frame['is_valid'], VALUE_COLUMN] = np.nan
n_missing = int(frame[VALUE_COLUMN].isna().sum())
if n_missing == 0:
    raise ValueError('The selected source recording has no missing or invalid samples; choose another CSV.')
print(f'Using {n_missing} genuine missing samples from {SOURCE_CSV.name}.')

Using 2005 genuine missing samples from S_1001_S1_TEX.csv.


In [4]:
config = DomainImputationConfig(
    domain='eye_tracking',
    timestamp_col='timestamp',
    timestamp_unit='ms',
    validity_col='is_valid',
    outlier_method='zscore',
    outlier_threshold=3.0,
    standardization_method='robust',
)

imputed_frame, provenance = impute_with_rf_selector(
    frame, VALUE_COLUMN, config=config,
    run_metadata=RunMetadata(
        executor_name=EXECUTOR_NAME, executor_info_path=EXECUTOR_INFO_PATH,
        executor_responsible_person=RESPONSIBLE_PERSON_NAME,
        executor_responsible_person_info_path=RESPONSIBLE_PERSON_INFO_PATH,
        execution_notebook_path=EXECUTION_NOTEBOOK_PATH,
        comment=f'Eye-tracking example from {SOURCE_CSV.name}.',
    ),
    input_path=SOURCE_CSV,
    output_path=OUTPUT_DIR / 'eyetracking_imputed.csv',
    provenance_path=OUTPUT_DIR / 'eyetracking_provenance.json',
)
provenance['summary']

{'gaps_before_outlier_detection': 17,
 'gaps_after_outlier_detection': 14,
 'filled_gap_count': 9,
 'skipped_gap_count': 5}

## Review

`eyetracking_imputed.csv` is the final data set; the two intermediate CSV files record the states after outlier detection and after imputation. `eyetracking_provenance.json` records the selected method, fallback behaviour, parameters, and file metadata. Review the summary before using the output in an analysis.